In [28]:
import os
import re
from pathlib import Path

# --- Configuration des chemins ---
ROOT_DIR = Path.cwd().absolute()
FILES_DIR = ROOT_DIR / "files" / "Anglais"
PAGES_DIR = ROOT_DIR / "pages"
TEMPLATE_PATH = ROOT_DIR / "template.html"

# Cible en_vocab.html directement dans le dossier pages/
EN_VOCAB_ROOT_PATH = PAGES_DIR / "en_vocab.html"

# Créer le dossier pages s'il n'existe pas
PAGES_DIR.mkdir(parents=True, exist_ok=True)

# Charger le modèle HTML
with open(TEMPLATE_PATH, "r", encoding="utf-8") as f:
    TEMPLATE_CONTENT = f.read()

def formater_nom_fichier(chaine):
    """Nettoie une chaîne pour en faire un nom de fichier web propre (slug)"""
    chaine = chaine.lower()
    chaine = re.sub(r'[^\w\s-]', '', chaine)
    return re.sub(r'[-\s]+', '-', chaine).strip('-')

def construire_chemin_page(segments, est_fiche=False, est_racine_vocab=False):
    """Génère le chemin complet du fichier HTML dans PAGES_DIR"""
    if est_racine_vocab:
        return EN_VOCAB_ROOT_PATH
        
    theme_slugs = [formater_nom_fichier(s) for s in segments[:-1]] if est_fiche else [formater_nom_fichier(s) for s in segments]
    
    # Remplacement du tiret par un tiret bas (_) pour marquer le changement de dossier
    nom_base = "en_vocab_theme-" + "_".join(theme_slugs)
    if est_fiche:
        nom_base += f"_liste-{formater_nom_fichier(segments[-1])}"
        
    return PAGES_DIR / f"{nom_base}.html"

def convertir_type(v_type):
    """Transforme les codes de types en libellés propres"""
    v_type = v_type.strip().upper()
    mapping = {
        'N': 'noun',
        'V': 'verb',
        'ADJ': 'adj.',
        'ABREV': 'abrev.',
        'A': 'other'
    }
    return mapping.get(v_type, v_type.lower())

def convertir_provenance(prov):
    """Transforme les codes de provenance en libellés propres"""
    prov = prov.strip().upper()
    mapping = {
        'LIVRE': 'Livre',
        'LIVREM': 'Marge du livre',
        'COURS': 'Vu en cours',
        'SUPPTHEME': 'Supplément du thème',
        'SUPP': 'Supplément'
    }
    return mapping.get(prov, prov)

def generer_breadcrumbs(segments, est_fiche=False, est_racine_vocab=False):
    """Construit le fil d'Ariane adapté à une exécution où toutes les pages sont dans /pages"""
    breadcrumbs = ['<a href="../index.html">Accueil</a>']
    
    if est_racine_vocab:
        breadcrumbs.append('<a href="./en_vocab.html">Vocabulaire</a>')
        return " > ".join(breadcrumbs)
        
    breadcrumbs.append('<a href="./en_vocab.html">Vocabulaire</a>')
    
    for i in range(len(segments)):
        current_segments = segments[:i+1]
        is_last = (i == len(segments) - 1)
        
        if is_last and est_fiche:
            target_file = construire_chemin_page(segments, est_fiche=True).name
        else:
            target_file = construire_chemin_page(current_segments, est_fiche=False).name
            
        titre_segment = segments[i].replace(";", " - ").replace("-", " ").capitalize()
        breadcrumbs.append(f'<a href="./{target_file}">{titre_segment}</a>')
        
    return " > ".join(breadcrumbs)

# --- 1. Cartographie de l'arborescence ---
arborescence = {}
themes_principaux = set()

for root, dirs, files in os.walk(FILES_DIR):
    path_root = Path(root)
    relative_parts = path_root.relative_to(FILES_DIR).parts
    
    if not relative_parts:
        for d in dirs:
            themes_principaux.add(d)
            if (d,) not in arborescence:
                arborescence[(d,)] = {'sous_themes': set(), 'fiches': set()}
        continue

    current_node = relative_parts
    if current_node not in arborescence:
        arborescence[current_node] = {'sous_themes': set(), 'fiches': set()}

    for d in dirs:
        arborescence[current_node]['sous_themes'].add(d)
        child_node = current_node + (d,)
        if child_node not in arborescence:
            arborescence[child_node] = {'sous_themes': set(), 'fiches': set()}
            
    for f in files:
        if f.endswith('.txt'):
            arborescence[current_node]['fiches'].add(f[:-4])

# --- 2. Génération des pages ---

# -- A. Génération de la page racine pages/en_vocab.html --
corps_racine = '<p class="intro-text">Ici les fiches sont triées par thèmes</p>\n'
corps_racine += '<section class="subject-section">\n<h3>Thèmes principaux</h3>\n<div class="topic-grid">\n'
for t in sorted(themes_principaux):
    lien_cible = f"./{construire_chemin_page((t,), est_fiche=False).name}"
    nom_image = f"en_vocab_theme-{formater_nom_fichier(t)}.png"
    
    corps_racine += f"""
    <a href="{lien_cible}" class="subject-topic">
        <span class="topic-title">{t.replace("-", " ").capitalize()}</span>
        <img src="../images/{nom_image}" alt="img {t}" class="topic-image">
    </a>\n"""
corps_racine += '</div>\n</section>\n'

html_racine = TEMPLATE_CONTENT.format(
    PAGE_TITLE="Vocabulaire",
    MAIN_TITLE="ANGLAIS > Vocabulaire",
    BREADCRUMBS=generer_breadcrumbs([], est_racine_vocab=True),
    CONTENT=corps_racine
)
with open(EN_VOCAB_ROOT_PATH, "w", encoding="utf-8") as f_out:
    f_out.write(html_racine)


# -- B. Génération des Fiches de Vocabulaire (Fichiers TXT à 7 colonnes) --
for node, data in arborescence.items():
    for fiche in data['fiches']:
        chemin_txt = FILES_DIR / os.path.join(*node) / f"{fiche}.txt"
        segments_fiche = list(node) + [fiche]
        fichier_html_cible = construire_chemin_page(segments_fiche, est_fiche=True)
        
        titre_propre = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
        
        # Structure de la table réorganisée selon votre demande
        tableau_html = '<table class="vocab-table">\n<thead>\n<tr>'
        tableau_html += '<th>Type</th><th>Anglais</th><th>Français</th><th>Niveau</th><th>Note</th><th>Exemple</th><th>Provenance</th>'
        tableau_html += '</tr>\n</thead>\n<tbody>\n'
        
        with open(chemin_txt, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"): 
                    continue  # Ignore les lignes vides ou les commentaires
                
                parts = [p.strip() for p in line.split(';')]
                
                # Sécurité stricte : si la ligne n'a pas exactement 7 colonnes, on la saute ou on la corrige
                if len(parts) < 7:
                    continue
                
                # Extraction des 7 colonnes fixes de votre fichier texte
                anglais = parts[0]
                francais = parts[1]
                note = parts[2]
                exemple = parts[3]
                v_type = parts[4]
                provenance = parts[5]
                niveau = parts[6]
                
                # Conversion des libellés avec vos fonctions existantes
                type_propre = convertir_type(v_type)
                provenance_propre = convertir_provenance(provenance)
                
                # Nettoyage des cellules pour l'affichage (remplace "..." par du vide pour les notes/exemples)
                note_propre = "" if note == "..." else note
                exemple_propre = "" if exemple == "..." else exemple
                
                # Génération des classes dynamiques pour le CSS
                type_classe = formater_nom_fichier(type_propre) # Nettoie le type (ex: "adj." devient "adj")
                niveau_classe = niveau.strip().lower()           # Nettoie le niveau (ex: "A1" devient "a1")

                # Génération des structures HTML individuelles avec les classes spécifiques
                cell_type = f"<span class='badge badge-type badge-type-{type_classe}'>{type_propre}</span>" if type_propre != "..." else ""
                cell_anglais = f"<span class='txt-anglais'>{anglais}</span>"
                cell_francais = f"<span class='txt-francais'>{francais}</span>"
                cell_level = f"<span class='badge badge-level badge-level-{niveau_classe}'>{niveau}</span>" if niveau != "..." else ""
                cell_note = f"<span class='txt-note'>{note_propre}</span>"
                cell_exemple = f"<span class='txt-exemple'>{exemple_propre}</span>"
                cell_prov = f"<span class='badge badge-prov'>{provenance_propre}</span>" if provenance_propre != "..." else ""
                
                # Ordre strict des colonnes dans le tableau HTML (doit matcher vos balises <th>)
                # Ordre actuel : Type (1) | Anglais (2) | Français (3) | Niveau (4) | Note (5) | Exemple (6) | Provenance (7)
                tableau_html += (
                    f"<tr>"
                    f"<td>{cell_type}</td>"
                    f"<td>{cell_anglais}</td>"
                    f"<td>{cell_francais}</td>"
                    f"<td>{cell_level}</td>"
                    f"<td>{cell_note}</td>"
                    f"<td>{cell_exemple}</td>"
                    f"<td>{cell_prov}</td>"
                    f"</tr>\n"
                )
        
        tableau_html += "</tbody>\n</table>"
        
        corps_page = f"""
        <p class="intro-text">Fiche de vocabulaire</p>
        <h3 class="fiche-title">{titre_propre}</h3>
        {tableau_html}
        """
        
        breadcrumbs_html = generer_breadcrumbs(segments_fiche, est_fiche=True)
        html_final = TEMPLATE_CONTENT.format(
            PAGE_TITLE=titre_propre,
            MAIN_TITLE="ANGLAIS > Vocabulaire",
            BREADCRUMBS=breadcrumbs_html,
            CONTENT=corps_page
        )
        
        with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
            f_out.write(html_final)


# -- C. Génération des pages de Navigation (Thèmes & Sous-thèmes) --
for node, data in arborescence.items():
    fichier_html_cible = construire_chemin_page(node, est_fiche=False)
    titre_navigation = node[-1].replace("-", " ").capitalize()
    
    corps_page = ""
    
    if data['sous_themes']:
        corps_page += '<section class="subject-section">\n<h3>Sous-thèmes disponibles</h3>\n<div class="topic-grid">\n'
        for st in sorted(data['sous_themes']):
            node_cible = node + (st,)
            lien_cible = f"./{construire_chemin_page(node_cible, est_fiche=False).name}"
            
            # Formatage de l'image dynamique en_vocab_theme-dossier1_dossier2.png
            theme_slugs = [formater_nom_fichier(s) for s in node_cible]
            nom_image = f"en_vocab_theme-{'_'.join(theme_slugs)}.png"
            
            corps_page += f"""
            <a href="{lien_cible}" class="subject-topic">
                <span class="topic-title">{st.replace("-", " ").capitalize()}</span>
                <img src="../images/{nom_image}" alt="img {st}" class="topic-image">
            </a>\n"""
        corps_page += '</div>\n</section>\n'
        
    if data['fiches']:
        corps_page += '<section class="fiches-section">\n<h3>Listes de vocabulaire</h3>\n<ul class="subject-list">\n'
        for fiche in sorted(data['fiches']):
            segments_fiche = list(node) + [fiche]
            lien_cible = f"./{construire_chemin_page(segments_fiche, est_fiche=True).name}"
            titre_fiche = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
            corps_page += f'<li><a href="{lien_cible}" class="subject-link">📄 {titre_fiche}</a></li>\n'
        corps_page += '</ul>\n</section>\n'
        
    breadcrumbs_html = generer_breadcrumbs(node, est_fiche=False)
    html_final = TEMPLATE_CONTENT.format(
        PAGE_TITLE=titre_navigation,
        MAIN_TITLE="ANGLAIS > Vocabulaire",
        BREADCRUMBS=breadcrumbs_html,
        CONTENT=corps_page
    )
    
    with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
        f_out.write(html_final)

print("Génération réussie ! Les types, provenances et structures d'images ont été convertis.")

Génération réussie ! Les types, provenances et structures d'images ont été convertis.
